In [0]:
%pip install lightgbm statsmodels

In [0]:
spark.conf.set("spark.sql.session.timeZone", "UTC")
from pyspark.sql import functions as F
import numpy as np, pandas as pd, lightgbm as lgb

semi = (spark.table("workspace.vol_project.silver_btc_5m")
    .withColumn("date", F.to_date("bucket"))
    .groupBy("date")
    .agg(F.sum(F.when(F.col("log_ret") < 0, F.col("log_ret") ** 2).otherwise(0)).alias("rs_neg")))

df = (spark.table("workspace.vol_project.gold_btc_daily_rv")
      .select("date", "rv", "daily_ret", "volume")
      .join(semi, "date")
      .toPandas())
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").set_index("date")

In [0]:
W, M = 7, 30

df["x_d"]          = np.log(df["rv"])
df["x_w"]          = np.log(df["rv"].rolling(W).mean())
df["x_m"]          = np.log(df["rv"].rolling(M).mean())
df["x_down_share"] = df["rs_neg"] / df["rv"]
df["x_ret"]        = df["daily_ret"]
df["x_abs_ret"]    = df["daily_ret"].abs()
df["x_vol_ratio"]  = np.log(df["volume"] / df["volume"].rolling(M).mean())
df["x_dow_next"]   = (df.index + pd.Timedelta(days=1)).dayofweek
df["y"]            = np.log(df["rv"]).shift(-1)

data = df.dropna()

HAR_F  = ["x_d", "x_w", "x_m"]
FULL_F = HAR_F + ["x_down_share", "x_ret", "x_abs_ret", "x_vol_ratio", "x_dow_next"]
origins = data.index[data.index >= "2024-01-01"]
print(len(data), len(origins))   # origins should be 1003

In [0]:
params = dict(n_estimators=300, learning_rate=0.03, num_leaves=15,
              min_child_samples=30, subsample=0.8, subsample_freq=1,
              colsample_bytree=0.8, random_state=42, verbose=-1)

def lgb_walk_forward(features, window=730, refit_every=7, holdout=90):
    preds, model, smear = {}, None, None
    for i, t in enumerate(origins):
        if i % refit_every == 0:
            train = data[data.index < t].iloc[-window:]
            fit_part, val_part = train.iloc[:-holdout], train.iloc[-holdout:]
            m_val = lgb.LGBMRegressor(**params).fit(fit_part[features], fit_part["y"])
            smear = np.mean(np.exp(val_part["y"] - m_val.predict(val_part[features])))
            model = lgb.LGBMRegressor(**params).fit(train[features], train["y"])
        preds[t] = np.exp(model.predict(data.loc[[t], features])[0]) * smear
    return pd.Series(preds)

lgbm_har  = lgb_walk_forward(HAR_F)
lgbm_full = lgb_walk_forward(FULL_F)

In [0]:
actual = np.exp(data.loc[origins, "y"])

def qlike(a, f):
    x = a / f
    return (x - np.log(x) - 1).mean()

print("LGBM (HAR features):", qlike(actual, lgbm_har))
print("LGBM (full):        ", qlike(actual, lgbm_full))
print("HAR rolling (bar):   0.2700")

last = data[data.index < origins[-1]].iloc[-730:]
m = lgb.LGBMRegressor(**params, importance_type="gain").fit(last[FULL_F], last["y"])
pd.Series(m.feature_importances_, index=FULL_F).sort_values(ascending=False)

In [0]:
fc = pd.concat([
    pd.DataFrame({"model": "lgbm_har_730",  "forecast": lgbm_har}),
    pd.DataFrame({"model": "lgbm_full_730", "forecast": lgbm_full}),
])
fc["actual"] = actual.reindex(fc.index).values
fc.index.name = "origin_date"
fc = fc.reset_index()
fc["target_date"] = (fc["origin_date"] + pd.Timedelta(days=1)).dt.date
fc = fc[["target_date", "model", "forecast", "actual"]]

model_list = ", ".join(f"'{m}'" for m in fc["model"].unique())

(spark.createDataFrame(fc)
    .withColumn("created_at", F.current_timestamp())
    .select("target_date", "model", "forecast", "actual", "created_at")
    .write.format("delta").mode("overwrite")
    .option("replaceWhere", f"model IN ({model_list})")
    .saveAsTable("workspace.vol_project.forecasts"))

In [0]:
import statsmodels.api as sm

data = data.copy()
data["x_weekend_next"] = data["x_dow_next"].isin([5, 6]).astype(int)

def ols_walk_forward(features, window=730):
    preds = {}
    for t in origins:
        train = data[data.index < t].iloc[-window:]
        m = sm.OLS(train["y"], sm.add_constant(train[features])).fit()
        smear = np.mean(np.exp(m.resid))
        x_t = sm.add_constant(data.loc[[t], features], has_constant="add")
        preds[t] = np.exp(m.predict(x_t).iloc[0]) * smear
    return pd.Series(preds)

HARX_F = HAR_F + ["x_down_share", "x_ret", "x_abs_ret", "x_vol_ratio", "x_weekend_next"]

har_smear = ols_walk_forward(HAR_F)
harx      = ols_walk_forward(HARX_F)
har_wknd  = ols_walk_forward(HAR_F + ["x_weekend_next"])

for name, f in [("HAR (lognormal corr.)", None), ("HAR (smearing)", har_smear),
                ("HAR + weekend", har_wknd), ("HAR-X linear", harx),
                ("LGBM full", lgbm_full)]:
    if f is None:
        print(f"{name:22s} 0.2700")
    else:
        print(f"{name:22s} {qlike(actual, f):.4f}   bias: {f.mean()/actual.mean():.3f}")

In [0]:
fc = pd.concat([
    pd.DataFrame({"model": "har_smear_730", "forecast": har_smear}),
    pd.DataFrame({"model": "har_wknd_730",  "forecast": har_wknd}),
    pd.DataFrame({"model": "harx_730",      "forecast": harx}),
])
fc["actual"] = actual.reindex(fc.index).values
fc.index.name = "origin_date"
fc = fc.reset_index()
fc["target_date"] = (fc["origin_date"] + pd.Timedelta(days=1)).dt.date
fc = fc[["target_date", "model", "forecast", "actual"]]

model_list = ", ".join(f"'{m}'" for m in fc["model"].unique())

(spark.createDataFrame(fc)
    .withColumn("created_at", F.current_timestamp())
    .select("target_date", "model", "forecast", "actual", "created_at")
    .write.format("delta").mode("overwrite")
    .option("replaceWhere", f"model IN ({model_list})")
    .saveAsTable("workspace.vol_project.forecasts"))